In [2]:
# MandiMitra: Relative Market Performance Prediction
# Step 1: Load and inspect the reviewed market-comparison dataset

import pandas as pd
import numpy as np
import duckdb

from pathlib import Path

ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_PATH = (
    ROOT / "data" / "processed" /
    "market_comparison_v2.parquet"
)

df = pd.read_parquet(DATA_PATH)

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDate range:")
print(df["Arrival_Date"].min(), "to", df["Arrival_Date"].max())

print("\nUnique commodities:", df["Commodity"].nunique())

print(
    "Unique market series:",
    df.groupby([
        "Commodity", "State", "District",
        "Market", "Variety", "Grade"
    ]).ngroups
)

display(df.head())

Dataset shape: (11286611, 15)

Columns:
['Commodity', 'Variety', 'Grade', 'Arrival_Date', 'State', 'District', 'Market', 'Modal_Price', 'Source_Records', 'Exclusion_Reason', 'Peer_Market_Count', 'Peer_Median_Price', 'Price_Difference', 'Premium_Discount_Pct', 'Price_Ratio']

Data types:
Commodity                   str
Variety                     str
Grade                       str
Arrival_Date             object
State                       str
District                    str
Market                      str
Modal_Price             float64
Source_Records            int64
Exclusion_Reason            str
Peer_Market_Count         int64
Peer_Median_Price       float64
Price_Difference        float64
Premium_Discount_Pct    float64
Price_Ratio             float64
dtype: object

Date range:
2021-01-01 to 2025-10-31

Unique commodities: 45
Unique market series: 32311


,Commodity,Variety,Grade,Arrival_Date,State,District,Market,Modal_Price,Source_Records,Exclusion_Reason,Peer_Market_Count,Peer_Median_Price,Price_Difference,Premium_Discount_Pct,Price_Ratio
0,Amaranthus,Other,FAQ,2021-01-01,Kerala,Alappuzha,Alappuzha,4050.0,1,NaN,21,2200.0,1850.0,84.090909,1.840909
1,Amaranthus,Other,FAQ,2021-01-01,Kerala,Alappuzha,Harippad,4000.0,1,NaN,21,2200.0,1800.0,81.818182,1.818182
2,Amaranthus,Other,FAQ,2021-01-01,Kerala,Alappuzha,Kayamkulam,4100.0,1,NaN,21,2200.0,1900.0,86.363636,1.863636
3,Amaranthus,Amaranthus,FAQ,2021-01-01,Kerala,Ernakulam,Angamaly,2500.0,1,NaN,12,2700.0,-200.0,-7.407407,0.925926
4,Amaranthus,Other,FAQ,2021-01-01,Kerala,Ernakulam,Piravam,2500.0,1,NaN,21,2200.0,300.0,13.636364,1.136364


In [3]:
# Step 2: Validate market-series observations

df["Arrival_Date"] = pd.to_datetime(
    df["Arrival_Date"],
    errors="coerce"
)

SERIES_KEYS = [
    "Commodity",
    "State",
    "District",
    "Market",
    "Variety",
    "Grade"
]

# Check whether each market series has at most one record per date
duplicate_count = df.duplicated(
    subset=SERIES_KEYS + ["Arrival_Date"]
).sum()

# Check essential fields
quality_checks = {
    "Invalid dates": df["Arrival_Date"].isna().sum(),
    "Missing modal prices": df["Modal_Price"].isna().sum(),
    "Non-positive modal prices": (df["Modal_Price"] <= 0).sum(),
    "Missing peer medians": df["Peer_Median_Price"].isna().sum(),
    "Non-positive peer medians": (df["Peer_Median_Price"] <= 0).sum(),
    "Missing relative premiums": df["Premium_Discount_Pct"].isna().sum(),
    "Invalid peer counts": (df["Peer_Market_Count"] < 3).sum(),
    "Duplicate market-date records": duplicate_count
}

display(
    pd.Series(quality_checks, name="Record_Count")
    .to_frame()
)

# Observations per market series
series_sizes = (
    df.groupby(SERIES_KEYS)
    .size()
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
)

display(series_sizes.to_frame("Observations_Per_Series"))

# Prepare chronological ordering
df = df.sort_values(
    SERIES_KEYS + ["Arrival_Date"]
).reset_index(drop=True)

,Record_Count
Invalid dates,0
Missing modal prices,0
Non-positive modal prices,0
Missing peer medians,0
Non-positive peer medians,0
Missing relative premiums,0
Invalid peer counts,0
Duplicate market-date records,0


,Observations_Per_Series
count,32311.000000
mean,349.311720
std,414.027763
min,1.000000
10%,3.000000
25%,27.000000
50%,177.000000
75%,474.000000
90%,1062.000000
max,1742.000000


In [4]:
# Step 3: Construct the future 7-day relative-performance target

con = duckdb.connect()

con.register("market_data", df)

target_data = con.execute("""
    WITH future_targets AS (

        SELECT
            *,

            AVG(Premium_Discount_Pct) OVER (
                PARTITION BY
                    Commodity,
                    State,
                    District,
                    Market,
                    Variety,
                    Grade
                ORDER BY Arrival_Date
                RANGE BETWEEN
                    INTERVAL 1 DAY FOLLOWING
                    AND INTERVAL 7 DAY FOLLOWING
            ) AS Target_7D_Avg_Premium,

            COUNT(*) OVER (
                PARTITION BY
                    Commodity,
                    State,
                    District,
                    Market,
                    Variety,
                    Grade
                ORDER BY Arrival_Date
                RANGE BETWEEN
                    INTERVAL 1 DAY FOLLOWING
                    AND INTERVAL 7 DAY FOLLOWING
            ) AS Future_Observation_Count

        FROM market_data
    )

    SELECT *
    FROM future_targets

    WHERE Arrival_Date <= DATE '2025-10-24'
      AND Future_Observation_Count >= 3

""").df()

print("Target dataset shape:", target_data.shape)

print(
    "Unique eligible market series:",
    target_data.groupby(SERIES_KEYS).ngroups
)

print(
    "Prediction date range:",
    target_data["Arrival_Date"].min(),
    "to",
    target_data["Arrival_Date"].max()
)

print("\nFuture observation count distribution:")

display(
    target_data["Future_Observation_Count"]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
    .to_frame("Future_Observations")
)

print("\nTarget distribution:")

display(
    target_data["Target_7D_Avg_Premium"]
    .describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
    .to_frame("Future_7D_Premium")
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Target dataset shape: (10183337, 17)
Unique eligible market series: 25751
Prediction date range: 2021-01-01 00:00:00 to 2025-10-24 00:00:00

Future observation count distribution:


,Future_Observations
count,1.018334e+07
mean,5.656205e+00
std,1.201805e+00
min,3.000000e+00
10%,4.000000e+00
25%,5.000000e+00
50%,6.000000e+00
75%,7.000000e+00
90%,7.000000e+00
max,7.000000e+00



Target distribution:


,Future_7D_Premium
count,1.018334e+07
mean,2.402477e+01
std,5.710795e+03
min,-9.987534e+01
1%,-7.028523e+01
5%,-5.113062e+01
25%,-1.914391e+01
50%,0.000000e+00
75%,2.530131e+01
95%,1.207487e+02


In [5]:
# Step 4: Investigate extreme future relative-performance targets

# Count observations beyond increasingly extreme thresholds
threshold_counts = {}

for threshold in [100, 200, 500, 1000, 10000]:
    threshold_counts[f"Above +{threshold}%"] = (
        target_data["Target_7D_Avg_Premium"] > threshold
    ).sum()

    threshold_counts[f"Below -{threshold}%"] = (
        target_data["Target_7D_Avg_Premium"] < -threshold
    ).sum()

display(
    pd.Series(threshold_counts, name="Observation_Count")
    .to_frame()
)

# Inspect the 30 largest positive target values
extreme_targets = (
    target_data.nlargest(
        30,
        "Target_7D_Avg_Premium"
    )[
        SERIES_KEYS
        + [
            "Arrival_Date",
            "Modal_Price",
            "Peer_Median_Price",
            "Premium_Discount_Pct",
            "Target_7D_Avg_Premium",
            "Future_Observation_Count"
        ]
    ]
)

display(extreme_targets)

# Inspect the most negative target values
lowest_targets = (
    target_data.nsmallest(
        20,
        "Target_7D_Avg_Premium"
    )[
        SERIES_KEYS
        + [
            "Arrival_Date",
            "Modal_Price",
            "Peer_Median_Price",
            "Premium_Discount_Pct",
            "Target_7D_Avg_Premium",
            "Future_Observation_Count"
        ]
    ]
)

display(lowest_targets)

,Observation_Count
Above +100%,688033
Below -100%,0
Above +200%,181008
Below -200%,0
Above +500%,19755
Below -500%,0
Above +1000%,10064
Below -1000%,0
Above +10000%,2074
Below -10000%,0


,Commodity,State,District,Market,Variety,Grade,Arrival_Date,Modal_Price,Peer_Median_Price,Premium_Discount_Pct,Target_7D_Avg_Premium,Future_Observation_Count
5338707,Onion,Maharashtra,Nashik,Kalvan,Red,FAQ,2024-12-20,1850.0,3250.0,-43.076923,1.026379e+07,3
5338706,Onion,Maharashtra,Nashik,Kalvan,Red,FAQ,2024-12-19,1800.0,3340.0,-46.107784,1.026379e+07,3
5338704,Onion,Maharashtra,Nashik,Kalvan,Red,FAQ,2024-12-16,2100.0,3420.0,-38.596491,7.697832e+06,4
5338705,Onion,Maharashtra,Nashik,Kalvan,Red,FAQ,2024-12-17,1900.0,3390.0,-43.952802,7.697832e+06,4
1295380,Mint (Pudina),Kerala,Alappuzha,Alappuzha,Other,FAQ,2021-12-18,9100.0,950.0,857.894737,2.190667e+05,3
1295341,Mint (Pudina),Kerala,Alappuzha,Alappuzha,Other,FAQ,2021-10-14,5950.0,4.0,148650.000000,1.982333e+05,4
1295313,Mint (Pudina),Kerala,Alappuzha,Alappuzha,Other,FAQ,2021-08-05,5950.0,550.0,981.818182,1.883679e+05,6
1295340,Mint (Pudina),Kerala,Alappuzha,Alappuzha,Other,FAQ,2021-10-13,6050.0,451.5,1239.977852,1.883167e+05,5
1295342,Mint (Pudina),Kerala,Alappuzha,Alappuzha,Other,FAQ,2021-10-17,5950.0,3.0,198233.333333,1.858375e+05,4
1295343,Mint (Pudina),Kerala,Alappuzha,Alappuzha,Other,FAQ,2021-10-18,5950.0,3.0,198233.333333,1.858375e+05,4


,Commodity,State,District,Market,Variety,Grade,Arrival_Date,Modal_Price,Peer_Median_Price,Premium_Discount_Pct,Target_7D_Avg_Premium,Future_Observation_Count
10145077,Mint (Pudina),Maharashtra,Pune,Pune,Other,FAQ,2021-03-15,2.0,650.0,-99.692308,-99.875335,3
10145137,Mint (Pudina),Maharashtra,Pune,Pune,Other,FAQ,2021-06-06,1.0,3.0,-66.666667,-99.869353,3
10145916,Mint (Pudina),Maharashtra,Pune,Pune,Other,FAQ,2024-04-08,3.0,1550.0,-99.806452,-99.859111,3
8039325,Mint (Pudina),Maharashtra,Pune,Pune (Manjri),Other,FAQ,2022-01-14,2.0,500.0,-99.600000,-99.854802,3
10145078,Mint (Pudina),Maharashtra,Pune,Pune,Other,FAQ,2021-03-16,1.0,1125.0,-99.911111,-99.853683,3
10145915,Mint (Pudina),Maharashtra,Pune,Pune,Other,FAQ,2024-04-07,2.0,905.0,-99.779006,-99.848649,3
8039319,Mint (Pudina),Maharashtra,Pune,Pune (Manjri),Other,FAQ,2022-01-04,2.0,1100.0,-99.818182,-99.846469,7
8039326,Mint (Pudina),Maharashtra,Pune,Pune (Manjri),Other,FAQ,2022-01-19,2.0,1500.0,-99.866667,-99.843691,3
8039320,Mint (Pudina),Maharashtra,Pune,Pune (Manjri),Other,FAQ,2022-01-05,2.0,1500.0,-99.866667,-99.843103,6
10145141,Mint (Pudina),Maharashtra,Pune,Pune,Other,FAQ,2021-06-14,3.0,825.0,-99.636364,-99.839707,6


In [6]:
# Step 5: Inspect future observations behind extreme targets

def inspect_future_window(
    commodity,
    state,
    district,
    market,
    start_date,
    variety,
    grade
):
    start_date = pd.Timestamp(start_date)
    end_date = start_date + pd.Timedelta(days=7)

    mask = (
        (df["Commodity"] == commodity) &
        (df["State"] == state) &
        (df["District"] == district) &
        (df["Market"] == market) &
        (df["Variety"] == variety) &
        (df["Grade"] == grade) &
        (df["Arrival_Date"] > start_date) &
        (df["Arrival_Date"] <= end_date)
    )

    result = df.loc[
        mask,
        [
            "Arrival_Date",
            "Modal_Price",
            "Peer_Median_Price",
            "Peer_Market_Count",
            "Premium_Discount_Pct"
        ]
    ].sort_values("Arrival_Date")

    print(f"Prediction date: {start_date.date()}")
    print(f"Future window: {start_date.date()} to {end_date.date()}")
    print("Future observations:", len(result))

    display(result)


# Case 1: Extreme Onion target
inspect_future_window(
    commodity="Onion",
    state="Maharashtra",
    district="Nashik",
    market="Kalvan",
    start_date="2024-12-20",
    variety="Red",
    grade="FAQ"
)

# Case 2: Persistently elevated Mint target
inspect_future_window(
    commodity="Mint (Pudina)",
    state="Kerala",
    district="Alappuzha",
    market="Alappuzha",
    start_date="2021-12-18",
    variety="Other",
    grade="FAQ"
)

Prediction date: 2024-12-20
Future window: 2024-12-20 to 2024-12-27
Future observations: 3


,Arrival_Date,Modal_Price,Peer_Median_Price,Peer_Market_Count,Premium_Discount_Pct
7417453,2024-12-23,917588483.0,2980.0,183,3.079146e+07
7417454,2024-12-24,1601.0,2880.0,155,-4.440972e+01
7417455,2024-12-27,1700.0,2600.0,184,-3.461538e+01


Prediction date: 2021-12-18
Future window: 2021-12-18 to 2021-12-25
Future observations: 3


,Arrival_Date,Modal_Price,Peer_Median_Price,Peer_Market_Count,Premium_Discount_Pct
6987326,2021-12-19,9100.0,4.0,5,227400.0
6987327,2021-12-20,9100.0,4.0,7,227400.0
6987328,2021-12-21,8100.0,4.0,7,202400.0


In [7]:
# Step 6: Compare alternative robust target definitions

robust_targets = con.execute("""
    SELECT
        *,

        COUNT(*) OVER (
            PARTITION BY
                Commodity, State, District, Market, Variety, Grade
            ORDER BY Arrival_Date
            RANGE BETWEEN
                INTERVAL 1 DAY FOLLOWING
                AND INTERVAL 7 DAY FOLLOWING
        ) AS Future_Observation_Count,

        AVG(Premium_Discount_Pct) OVER (
            PARTITION BY
                Commodity, State, District, Market, Variety, Grade
            ORDER BY Arrival_Date
            RANGE BETWEEN
                INTERVAL 1 DAY FOLLOWING
                AND INTERVAL 7 DAY FOLLOWING
        ) AS Target_7D_Avg_Premium,

        MEDIAN(Premium_Discount_Pct) OVER (
            PARTITION BY
                Commodity, State, District, Market, Variety, Grade
            ORDER BY Arrival_Date
            RANGE BETWEEN
                INTERVAL 1 DAY FOLLOWING
                AND INTERVAL 7 DAY FOLLOWING
        ) AS Target_7D_Median_Premium,

        MEDIAN(
            LN(Modal_Price / Peer_Median_Price)
        ) OVER (
            PARTITION BY
                Commodity, State, District, Market, Variety, Grade
            ORDER BY Arrival_Date
            RANGE BETWEEN
                INTERVAL 1 DAY FOLLOWING
                AND INTERVAL 7 DAY FOLLOWING
        ) AS Target_7D_Median_Log_Ratio

    FROM market_data

    QUALIFY
        Arrival_Date <= DATE '2025-10-24'
        AND Future_Observation_Count >= 3

""").df()

# Compare target distributions
target_comparison = robust_targets[
    [
        "Target_7D_Avg_Premium",
        "Target_7D_Median_Premium",
        "Target_7D_Median_Log_Ratio"
    ]
].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

display(target_comparison)

# Inspect the two previously investigated cases
case_mask = (
    (
        (robust_targets["Commodity"] == "Onion") &
        (robust_targets["Market"] == "Kalvan") &
        (robust_targets["Arrival_Date"] == pd.Timestamp("2024-12-20"))
    )
    |
    (
        (robust_targets["Commodity"] == "Mint (Pudina)") &
        (robust_targets["Market"] == "Alappuzha") &
        (robust_targets["Arrival_Date"] == pd.Timestamp("2021-12-18"))
    )
)

display(
    robust_targets.loc[
        case_mask,
        SERIES_KEYS + [
            "Arrival_Date",
            "Target_7D_Avg_Premium",
            "Target_7D_Median_Premium",
            "Target_7D_Median_Log_Ratio",
            "Future_Observation_Count"
        ]
    ]
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
Target_7D_Avg_Premium,10183337.0,24.024773,5710.794540,-99.875335,-70.285229,-51.130616,-19.143913,0.0,25.301312,120.748722,252.531312,1.026379e+07
Target_7D_Median_Premium,10183337.0,15.960948,521.147892,-99.906977,-71.250000,-52.000000,-20.000000,0.0,25.000000,120.000000,246.925134,2.682333e+05
Target_7D_Median_Log_Ratio,10183337.0,-0.003868,0.524357,-6.980076,-1.248575,-0.733969,-0.223144,0.0,0.223144,0.788457,1.243194,7.894815e+00


,Commodity,State,District,Market,Variety,Grade,Arrival_Date,Target_7D_Avg_Premium,Target_7D_Median_Premium,Target_7D_Median_Log_Ratio,Future_Observation_Count
1334229,Mint (Pudina),Kerala,Alappuzha,Alappuzha,Other,FAQ,2021-12-18,2.190667e+05,227400.000000,7.729735,3
5338961,Onion,Maharashtra,Nashik,Kalvan,Red,FAQ,2024-12-20,1.026379e+07,-34.615385,-0.424883,3


In [8]:
# Step 7: Compare target quality across commodities

commodity_target_audit = (
    robust_targets
    .groupby("Commodity")
    .agg(
        Prediction_Opportunities=(
            "Target_7D_Median_Log_Ratio", "count"
        ),
        Median_Log_Ratio=(
            "Target_7D_Median_Log_Ratio", "median"
        ),
        Mean_Log_Ratio=(
            "Target_7D_Median_Log_Ratio", "mean"
        ),
        Std_Log_Ratio=(
            "Target_7D_Median_Log_Ratio", "std"
        ),
        P01_Log_Ratio=(
            "Target_7D_Median_Log_Ratio",
            lambda x: x.quantile(0.01)
        ),
        P99_Log_Ratio=(
            "Target_7D_Median_Log_Ratio",
            lambda x: x.quantile(0.99)
        ),
        Extreme_Target_Pct=(
            "Target_7D_Median_Log_Ratio",
            lambda x: 100 * (x.abs() > 2).mean()
        )
    )
    .reset_index()
    .sort_values(
        "Extreme_Target_Pct",
        ascending=False
    )
)

display(
    commodity_target_audit.round(3)
)

,Commodity,Prediction_Opportunities,Median_Log_Ratio,Mean_Log_Ratio,Std_Log_Ratio,P01_Log_Ratio,P99_Log_Ratio,Extreme_Target_Pct
24,Methi (Leaves),52359,-0.017,-0.737,2.064,-5.395,1.867,20.060
32,Spinach,125482,0.000,-0.381,1.498,-5.421,1.329,8.949
27,Onion Green,97106,0.000,-0.193,1.194,-4.938,1.766,6.873
25,Mint (Pudina),79655,0.000,-0.135,1.327,-5.807,3.672,6.280
22,Green Peas,36085,0.000,0.057,0.513,-1.145,1.898,0.809
15,Drumstick,159238,0.000,-0.049,0.427,-1.476,0.835,0.563
10,Carrot,344858,0.000,0.074,0.588,-1.190,1.792,0.300
1,Beans,104392,0.000,-0.019,0.327,-1.012,0.836,0.258
19,Garlic,276937,0.000,-0.037,0.390,-1.306,0.847,0.202
9,Capsicum,212393,0.000,-0.023,0.502,-1.426,1.264,0.155


In [9]:
# Step 8: Identify market series with extreme future relative-price targets

problem_commodities = [
    "Methi (Leaves)",
    "Spinach",
    "Onion Green",
    "Mint (Pudina)"
]

leafy_series_audit = (
    robust_targets[
        robust_targets["Commodity"].isin(problem_commodities)
    ]
    .groupby(SERIES_KEYS)
    .agg(
        Prediction_Opportunities=(
            "Target_7D_Median_Log_Ratio", "count"
        ),
        Median_Log_Ratio=(
            "Target_7D_Median_Log_Ratio", "median"
        ),
        Mean_Log_Ratio=(
            "Target_7D_Median_Log_Ratio", "mean"
        ),
        Extreme_Target_Pct=(
            "Target_7D_Median_Log_Ratio",
            lambda x: 100 * (x.abs() > 2).mean()
        )
    )
    .reset_index()
)

leafy_series_audit = leafy_series_audit[
    leafy_series_audit["Prediction_Opportunities"] >= 30
].sort_values(
    ["Commodity", "Extreme_Target_Pct"],
    ascending=[True, False]
)

display(
    leafy_series_audit.round(3)
)

,Commodity,State,District,Market,Variety,Grade,Prediction_Opportunities,Median_Log_Ratio,Mean_Log_Ratio,Extreme_Target_Pct
101,Methi (Leaves),Maharashtra,Ahmednagar,Rahata,Other,Local,201,-4.541,-4.576,100.0
102,Methi (Leaves),Maharashtra,Ahmednagar,Rahuri,Other,FAQ,76,-4.835,-4.815,100.0
103,Methi (Leaves),Maharashtra,Ahmednagar,Rahuri,Other,Local,33,-4.711,-4.684,100.0
105,Methi (Leaves),Maharashtra,Ahmednagar,Shrirampur,Other,Local,207,-4.574,-4.624,100.0
132,Methi (Leaves),Maharashtra,Nashik,Chandvad,Other,FAQ,435,-4.750,-4.753,100.0
...,...,...,...,...,...,...,...,...,...,...
1173,Spinach,Uttar Pradesh,Saharanpur,Saharanpur,Spinach,FAQ,227,0.038,0.006,0.0
1174,Spinach,Uttar Pradesh,Sambhal,Chandausi,Spinach,FAQ,90,0.012,-0.008,0.0
1175,Spinach,Uttar Pradesh,Unnao,Unnao,Spinach,FAQ,664,0.041,0.062,0.0
1176,Spinach,Uttar Pradesh,Varanasi,Varanasi,Spinach,FAQ,79,-0.075,-0.083,0.0


In [10]:
# Step 9: Summarise persistent extreme-target series

leafy_series_summary = (
    leafy_series_audit
    .groupby("Commodity")
    .agg(
        Total_Series=("Market", "count"),
        Series_With_At_Least_50_Pct_Extremes=(
            "Extreme_Target_Pct",
            lambda x: (x >= 50).sum()
        ),
        Series_With_At_Least_80_Pct_Extremes=(
            "Extreme_Target_Pct",
            lambda x: (x >= 80).sum()
        ),
        Series_With_At_Least_95_Pct_Extremes=(
            "Extreme_Target_Pct",
            lambda x: (x >= 95).sum()
        ),
        Median_Extreme_Target_Pct=(
            "Extreme_Target_Pct", "median"
        )
    )
    .reset_index()
)

display(leafy_series_summary)

,Commodity,Total_Series,Series_With_At_Least_50_Pct_Extremes,Series_With_At_Least_80_Pct_Extremes,Series_With_At_Least_95_Pct_Extremes,Median_Extreme_Target_Pct
0,Methi (Leaves),174,21,20,20,0.0
1,Mint (Pudina),187,6,6,5,0.0
2,Onion Green,233,15,15,13,0.0
3,Spinach,283,20,20,20,0.0


In [11]:
# Step 10: Inspect persistently extreme market series

persistent_extreme_series = leafy_series_audit[
    leafy_series_audit["Extreme_Target_Pct"] >= 80
].copy()

display(
    persistent_extreme_series[
        SERIES_KEYS + [
            "Prediction_Opportunities",
            "Median_Log_Ratio",
            "Mean_Log_Ratio",
            "Extreme_Target_Pct"
        ]
    ]
    .sort_values(
        ["Commodity", "Extreme_Target_Pct"],
        ascending=[True, False]
    )
    .round(3)
)

print(
    "Total persistently extreme series:",
    len(persistent_extreme_series)
)

,Commodity,State,District,Market,Variety,Grade,Prediction_Opportunities,Median_Log_Ratio,Mean_Log_Ratio,Extreme_Target_Pct
101,Methi (Leaves),Maharashtra,Ahmednagar,Rahata,Other,Local,201,-4.541,-4.576,100.000
102,Methi (Leaves),Maharashtra,Ahmednagar,Rahuri,Other,FAQ,76,-4.835,-4.815,100.000
103,Methi (Leaves),Maharashtra,Ahmednagar,Rahuri,Other,Local,33,-4.711,-4.684,100.000
105,Methi (Leaves),Maharashtra,Ahmednagar,Shrirampur,Other,Local,207,-4.574,-4.624,100.000
132,Methi (Leaves),Maharashtra,Nashik,Chandvad,Other,FAQ,435,-4.750,-4.753,100.000
...,...,...,...,...,...,...,...,...,...,...
1017,Spinach,Maharashtra,Sholapur,Akluj,Other,Local,172,-4.427,-4.436,100.000
1018,Spinach,Maharashtra,Sholapur,Pandharpur,Other,FAQ,127,-5.452,-5.445,100.000
1021,Spinach,Maharashtra,Thane,Kalyan,Other,FAQ,610,-4.628,-4.625,100.000
973,Spinach,Maharashtra,Ahmednagar,Shrirampur,Other,FAQ,1081,-5.208,-5.218,99.815


Total persistently extreme series: 61


In [15]:
# Step 11: Historical-only comparability audit at 2023-12-31

historical_cutoff = pd.Timestamp("2023-12-31")

persistent_keys = persistent_extreme_series[
    SERIES_KEYS
].drop_duplicates()

historical_ratio_audit = (
    df.merge(
        persistent_keys,
        on=SERIES_KEYS,
        how="inner"
    )
)

historical_ratio_audit = historical_ratio_audit[
    historical_ratio_audit["Arrival_Date"] <= historical_cutoff
].copy()

historical_ratio_summary = (
    historical_ratio_audit
    .groupby(SERIES_KEYS)
    .agg(
        Historical_Observations=(
            "Premium_Discount_Pct", "count"
        ),
        Historical_Median_Log_Ratio=(
            "Price_Ratio",
            lambda x: np.log(x).median()
        ),
        Historical_Mean_Log_Ratio=(
            "Price_Ratio",
            lambda x: np.log(x).mean()
        ),
        Historical_Extreme_Pct=(
            "Price_Ratio",
            lambda x: 100 * (np.log(x).abs() > 2).mean()
        ),
        Historical_First_Date=("Arrival_Date", "min"),
        Historical_Last_Date=("Arrival_Date", "max")
    )
    .reset_index()
)

historical_ratio_summary = historical_ratio_summary.merge(
    persistent_extreme_series[
        SERIES_KEYS + ["Extreme_Target_Pct"]
    ],
    on=SERIES_KEYS,
    how="left"
)

display(
    historical_ratio_summary
    .sort_values(
        "Historical_Extreme_Pct",
        ascending=False
    )
    .round(3)
)

print(
    "Series with at least 100 historical observations:",
    (
        historical_ratio_summary[
            "Historical_Observations"
        ] >= 100
    ).sum()
)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8004\1239763365.py:60: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .round(3)


,Commodity,State,District,Market,Variety,Grade,Historical_Observations,Historical_Median_Log_Ratio,Historical_Mean_Log_Ratio,Historical_Extreme_Pct,Historical_First_Date,Historical_Last_Date,Extreme_Target_Pct
27,Spinach,Maharashtra,Pune,Pune,Other,FAQ,847,-5.011,-5.003,100.000,2021-01-01,2023-12-31,100.000
29,Spinach,Maharashtra,Pune,Pune (Manjri),Other,FAQ,826,-5.298,-5.299,100.000,2021-01-02,2023-12-31,100.000
7,Methi (Leaves),Maharashtra,Sholapur,Akluj,Other,FAQ,7,-4.748,-4.665,100.000,2023-12-21,2023-12-30,100.000
34,Spinach,Maharashtra,Thane,Kalyan,Other,FAQ,507,-4.646,-4.604,100.000,2021-01-01,2023-12-27,100.000
26,Spinach,Maharashtra,Ahmednagar,Shrirampur,Other,FAQ,821,-5.298,-5.244,100.000,2021-01-01,2023-12-30,99.815
30,Spinach,Maharashtra,Pune,Pune (Moshi),Other,FAQ,864,-5.060,-5.040,100.000,2021-01-01,2023-12-31,100.000
31,Spinach,Maharashtra,Pune,Pune (Pimpri),Other,FAQ,894,-4.856,-4.882,100.000,2021-01-01,2023-12-31,100.000
28,Spinach,Maharashtra,Pune,Pune (Khadiki),Other,FAQ,895,-5.043,-5.042,99.888,2021-01-01,2023-12-31,100.000
32,Spinach,Maharashtra,Sholapur,Akluj,Other,FAQ,619,-5.116,-5.129,99.838,2021-01-01,2023-12-31,100.000
24,Spinach,Maharashtra,Ahmednagar,Rahata,Other,FAQ,552,-5.011,-5.016,99.819,2021-01-14,2023-12-31,100.000


Series with at least 100 historical observations: 32


In [16]:
# Step 12: Apply a historical-only comparability screening rule

MIN_HISTORICAL_OBSERVATIONS = 100
EXTREME_RATIO_THRESHOLD = 2
EXTREME_FREQUENCY_THRESHOLD = 80

historical_ratio_summary["Historical_Quality_Flag"] = (
    (
        historical_ratio_summary["Historical_Observations"]
        >= MIN_HISTORICAL_OBSERVATIONS
    )
    &
    (
        historical_ratio_summary["Historical_Extreme_Pct"]
        >= EXTREME_FREQUENCY_THRESHOLD
    )
)

historical_ratio_summary["Historical_Eligibility"] = np.where(
    historical_ratio_summary["Historical_Observations"]
    < MIN_HISTORICAL_OBSERVATIONS,
    "Insufficient history",
    np.where(
        historical_ratio_summary["Historical_Quality_Flag"],
        "Flagged for comparability review",
        "Passes initial screen"
    )
)

display(
    historical_ratio_summary[
        "Historical_Eligibility"
    ].value_counts().to_frame("Series_Count")
)

display(
    historical_ratio_summary[
        [
            "Commodity",
            "State",
            "District",
            "Market",
            "Historical_Observations",
            "Historical_Median_Log_Ratio",
            "Historical_Extreme_Pct",
            "Historical_Eligibility"
        ]
    ]
    .sort_values(
        ["Historical_Eligibility", "Historical_Extreme_Pct"],
        ascending=[True, False]
    )
    .round(3)
)

,Series_Count
Historical_Eligibility,
Flagged for comparability review,32
Insufficient history,3


,Commodity,State,District,Market,Historical_Observations,Historical_Median_Log_Ratio,Historical_Extreme_Pct,Historical_Eligibility
26,Spinach,Maharashtra,Ahmednagar,Shrirampur,821,-5.298,100.000,Flagged for comparability review
27,Spinach,Maharashtra,Pune,Pune,847,-5.011,100.000,Flagged for comparability review
29,Spinach,Maharashtra,Pune,Pune (Manjri),826,-5.298,100.000,Flagged for comparability review
30,Spinach,Maharashtra,Pune,Pune (Moshi),864,-5.060,100.000,Flagged for comparability review
31,Spinach,Maharashtra,Pune,Pune (Pimpri),894,-4.856,100.000,Flagged for comparability review
34,Spinach,Maharashtra,Thane,Kalyan,507,-4.646,100.000,Flagged for comparability review
28,Spinach,Maharashtra,Pune,Pune (Khadiki),895,-5.043,99.888,Flagged for comparability review
32,Spinach,Maharashtra,Sholapur,Akluj,619,-5.116,99.838,Flagged for comparability review
24,Spinach,Maharashtra,Ahmednagar,Rahata,552,-5.011,99.819,Flagged for comparability review
33,Spinach,Maharashtra,Sholapur,Pandharpur,210,-5.397,99.524,Flagged for comparability review


In [17]:
# Step 13: Historical comparability audit for all market series

historical_comparability_audit = (
    df[df["Arrival_Date"] <= historical_cutoff]
    .assign(
        Log_Price_Ratio=lambda x: np.log(
            x["Price_Ratio"]
        )
    )
    .groupby(SERIES_KEYS)
    .agg(
        Historical_Observations=(
            "Log_Price_Ratio", "count"
        ),
        Historical_Median_Log_Ratio=(
            "Log_Price_Ratio", "median"
        ),
        Historical_Extreme_Pct=(
            "Log_Price_Ratio",
            lambda x: 100 * (x.abs() > 2).mean()
        ),
        Historical_First_Date=("Arrival_Date", "min"),
        Historical_Last_Date=("Arrival_Date", "max")
    )
    .reset_index()
)

historical_comparability_audit["Historical_Eligibility"] = np.select(
    [
        historical_comparability_audit[
            "Historical_Observations"
        ] < 100,

        historical_comparability_audit[
            "Historical_Extreme_Pct"
        ] >= 80
    ],
    [
        "Insufficient history",
        "Flagged for comparability review"
    ],
    default="Passes initial screen"
)

display(
    historical_comparability_audit[
        "Historical_Eligibility"
    ].value_counts().to_frame("Series_Count")
)

display(
    historical_comparability_audit
    .groupby(
        ["Commodity", "Historical_Eligibility"]
    )
    .size()
    .unstack(fill_value=0)
)

,Series_Count
Historical_Eligibility,
Passes initial screen,11490
Insufficient history,6945
Flagged for comparability review,37


Historical_Eligibility,Flagged for comparability review,Insufficient history,Passes initial screen
Commodity,,,
Amaranthus,0,31,67
Beans,0,76,65
Beetroot,0,83,137
Bhindi (Ladies Finger),0,264,675
Bitter gourd,0,320,544
Bottle gourd,0,223,506
Brinjal,0,365,864
Bunch Beans,0,11,0
Cabbage,0,301,630


In [18]:
# Step 14: Future target availability among historically eligible series

eligible_keys_2023 = historical_comparability_audit.loc[
    historical_comparability_audit["Historical_Eligibility"]
    == "Passes initial screen",
    SERIES_KEYS
].drop_duplicates()

eligible_targets = robust_targets.merge(
    eligible_keys_2023,
    on=SERIES_KEYS,
    how="inner"
)

eligible_targets["Prediction_Year"] = (
    eligible_targets["Arrival_Date"].dt.year
)

display(
    eligible_targets.groupby("Prediction_Year").agg(
        Prediction_Opportunities=(
            "Target_7D_Median_Log_Ratio", "count"
        ),
        Eligible_Market_Series=(
            SERIES_KEYS[0], "nunique"
        ),
        Median_Future_Observations=(
            "Future_Observation_Count", "median"
        )
    )
)

display(
    eligible_targets.groupby("Commodity").agg(
        Prediction_Opportunities=(
            "Target_7D_Median_Log_Ratio", "count"
        ),
        Eligible_Market_Series=(
            "Market", "nunique"
        )
    ).sort_values(
        "Prediction_Opportunities", ascending=False
    )
)

,Prediction_Opportunities,Eligible_Market_Series,Median_Future_Observations
Prediction_Year,,,
2021,1573667,40,6.0
2022,1812942,40,6.0
2023,1778787,40,6.0
2024,1613032,40,6.0
2025,1116517,40,6.0


,Prediction_Opportunities,Eligible_Market_Series
Commodity,,
Potato,770546,886
Onion,748788,880
Tomato,724838,876
Brinjal,686990,826
Green Chilli,608524,723
Cauliflower,441135,676
Bhindi (Ladies Finger),428935,670
Cabbage,413470,623
Bottle gourd,395081,503


In [19]:
# Step 15: Correct yearly market-series coverage

eligible_targets["Series_ID"] = (
    eligible_targets
    .groupby(SERIES_KEYS, sort=False)
    .ngroup()
)

yearly_target_coverage = (
    eligible_targets
    .groupby("Prediction_Year")
    .agg(
        Prediction_Opportunities=(
            "Target_7D_Median_Log_Ratio", "count"
        ),
        Eligible_Market_Series=(
            "Series_ID", "nunique"
        ),
        Median_Future_Observations=(
            "Future_Observation_Count", "median"
        )
    )
)

display(yearly_target_coverage)

# Focus on our intended validation and test periods

display(
    yearly_target_coverage.loc[
        [2024, 2025]
    ]
)

,Prediction_Opportunities,Eligible_Market_Series,Median_Future_Observations
Prediction_Year,,,
2021,1573667,9803,6.0
2022,1812942,10671,6.0
2023,1778787,10619,6.0
2024,1613032,9105,6.0
2025,1116517,8451,6.0


,Prediction_Opportunities,Eligible_Market_Series,Median_Future_Observations
Prediction_Year,,,
2024,1613032,9105,6.0
2025,1116517,8451,6.0


In [20]:
# Step 16: Build leakage-safe historical features

target_labels = eligible_targets[
    SERIES_KEYS
    + [
        "Arrival_Date",
        "Target_7D_Median_Log_Ratio",
        "Future_Observation_Count"
    ]
].copy()

con.register("target_labels", target_labels)

model_data = con.execute("""
    WITH historical_features AS (
        SELECT
            Commodity,
            State,
            District,
            Market,
            Variety,
            Grade,
            Arrival_Date,

            LN(Price_Ratio) AS Current_Log_Ratio,

            LAG(LN(Price_Ratio), 1) OVER w
                AS Lag_1_Log_Ratio,

            LAG(LN(Price_Ratio), 3) OVER w
                AS Lag_3_Log_Ratio,

            LAG(LN(Price_Ratio), 7) OVER w
                AS Lag_7_Log_Ratio,

            MEDIAN(LN(Price_Ratio)) OVER (
                PARTITION BY Commodity, State, District,
                             Market, Variety, Grade
                ORDER BY Arrival_Date
                ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
            ) AS Rolling_Median_7,

            MEDIAN(LN(Price_Ratio)) OVER (
                PARTITION BY Commodity, State, District,
                             Market, Variety, Grade
                ORDER BY Arrival_Date
                ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
            ) AS Rolling_Median_30,

            AVG(LN(Price_Ratio)) OVER (
                PARTITION BY Commodity, State, District,
                             Market, Variety, Grade
                ORDER BY Arrival_Date
                ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
            ) AS Rolling_Mean_7,

            STDDEV_SAMP(LN(Price_Ratio)) OVER (
                PARTITION BY Commodity, State, District,
                             Market, Variety, Grade
                ORDER BY Arrival_Date
                ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
            ) AS Rolling_Std_7,

            Modal_Price,
            LAG(Modal_Price, 1) OVER w AS Previous_Price,
            LAG(Modal_Price, 7) OVER w AS Price_7_Obs_Ago,

            DATEDIFF(
                'day',
                LAG(Arrival_Date, 1) OVER w,
                Arrival_Date
            ) AS Days_Since_Previous_Report,

            Peer_Market_Count,
            Source_Records,

            MONTH(Arrival_Date) AS Month,
            DAYOFYEAR(Arrival_Date) AS Day_Of_Year

        FROM market_data

        WINDOW w AS (
            PARTITION BY Commodity, State, District,
                         Market, Variety, Grade
            ORDER BY Arrival_Date
        )
    ),

    features AS (
        SELECT
            *,
            LN(Modal_Price / Previous_Price)
                AS Log_Price_Change_1,

            LN(Modal_Price / Price_7_Obs_Ago)
                AS Log_Price_Change_7

        FROM historical_features
    )

    SELECT
        t.*,
        f.Current_Log_Ratio,
        f.Lag_1_Log_Ratio,
        f.Lag_3_Log_Ratio,
        f.Lag_7_Log_Ratio,
        f.Rolling_Median_7,
        f.Rolling_Median_30,
        f.Rolling_Mean_7,
        f.Rolling_Std_7,
        f.Log_Price_Change_1,
        f.Log_Price_Change_7,
        f.Days_Since_Previous_Report,
        f.Peer_Market_Count,
        f.Source_Records,
        f.Month,
        f.Day_Of_Year

    FROM target_labels t
    INNER JOIN features f
        USING (
            Commodity, State, District, Market,
            Variety, Grade, Arrival_Date
        )

    ORDER BY Arrival_Date
""").df()

print("Model dataset shape:", model_data.shape)
print("\nDate range:")
print(model_data["Arrival_Date"].min(), "to",
      model_data["Arrival_Date"].max())

print("\nTarget summary:")
display(
    model_data["Target_7D_Median_Log_Ratio"].describe()
)

print("\nMissing feature values:")
display(model_data.isna().sum().to_frame("Missing_Count"))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Model dataset shape: (7894945, 24)

Date range:
2021-01-01 00:00:00 to 2025-10-24 00:00:00

Target summary:


count    7.894945e+06
mean     1.562761e-02
std      4.748121e-01
min     -5.560682e+00
25%     -2.344385e-01
50%      0.000000e+00
75%      2.513144e-01
max      6.996529e+00
Name: Target_7D_Median_Log_Ratio, dtype: float64


Missing feature values:


,Missing_Count
Commodity,0
State,0
District,0
Market,0
Variety,0
Grade,0
Arrival_Date,0
Target_7D_Median_Log_Ratio,0
Future_Observation_Count,0
Current_Log_Ratio,0


In [21]:
# Step 17: Chronological splits and baseline evaluation

from sklearn.metrics import mean_absolute_error, mean_squared_error

model_data["Split"] = np.select(
    [
        model_data["Arrival_Date"] < "2024-01-01",
        model_data["Arrival_Date"] < "2025-01-01"
    ],
    [
        "Train",
        "Validation"
    ],
    default="Test"
)

BASELINES = {
    "Peer Parity": "zero",
    "Current Ratio": "Current_Log_Ratio",
    "Historical Median 30": "Rolling_Median_30"
}

baseline_results = []

for split_name in ["Train", "Validation", "Test"]:

    split_data = model_data[
        model_data["Split"] == split_name
    ]

    y_true = split_data["Target_7D_Median_Log_Ratio"]

    for baseline_name, prediction_column in BASELINES.items():

        if prediction_column == "zero":
            y_pred = np.zeros(len(split_data))
        else:
            y_pred = split_data[prediction_column]

        baseline_results.append({
            "Split": split_name,
            "Baseline": baseline_name,
            "Observations": len(split_data),
            "MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "Median_Absolute_Error": np.median(
                np.abs(y_true - y_pred)
            )
        })

baseline_results = pd.DataFrame(baseline_results)

display(
    baseline_results.round(4)
)

,Split,Baseline,Observations,MAE,RMSE,Median_Absolute_Error
0,Train,Peer Parity,5165396,0.3488,0.4817,0.2485
1,Train,Current Ratio,5165396,0.1128,0.1954,0.0664
2,Train,Historical Median 30,5165396,0.1600,0.2414,0.1051
3,Validation,Peer Parity,1613032,0.3252,0.4521,0.2281
4,Validation,Current Ratio,1613032,0.1098,0.1875,0.0645
5,Validation,Historical Median 30,1613032,0.1550,0.2330,0.1001
6,Test,Peer Parity,1116517,0.3410,0.4766,0.2360
7,Test,Current Ratio,1116517,0.1037,0.1668,0.0606
8,Test,Historical Median 30,1116517,0.1451,0.2166,0.0929


In [27]:
# Step 18: Prepare data and train Random Forest

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

FEATURE_COLUMNS = [
    "Current_Log_Ratio",
    "Lag_1_Log_Ratio",
    "Lag_3_Log_Ratio",
    "Lag_7_Log_Ratio",
    "Rolling_Median_7",
    "Rolling_Median_30",
    "Rolling_Mean_7",
    "Rolling_Std_7",
    "Log_Price_Change_1",
    "Log_Price_Change_7",
    "Days_Since_Previous_Report",
    "Peer_Market_Count",
    "Source_Records",
    "Month",
    "Day_Of_Year"
]

TARGET_COLUMN = "Target_7D_Median_Log_Ratio"

train_data = model_data[model_data["Split"] == "Train"]
validation_data = model_data[model_data["Split"] == "Validation"]

X_train = train_data[FEATURE_COLUMNS]
y_train = train_data[TARGET_COLUMN]

X_val = validation_data[FEATURE_COLUMNS]
y_val = validation_data[TARGET_COLUMN]

rf_model = RandomForestRegressor(
    n_estimators=150,
    max_depth=18,
    min_samples_leaf=10,
    max_features=0.8,
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_val_predictions = rf_model.predict(X_val)

rf_validation_results = {
    "Model": "Random Forest",
    "MAE": mean_absolute_error(
        y_val, rf_val_predictions
    ),
    "RMSE": np.sqrt(
        mean_squared_error(y_val, rf_val_predictions)
    )
}

display(pd.DataFrame([rf_validation_results]).round(4))

,Model,MAE,RMSE
0,Random Forest,0.0964,0.152


In [28]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

validation_results = validation_data[
    SERIES_KEYS + [TARGET_COLUMN]
].copy()

validation_results["RF_Prediction"] = rf_val_predictions
validation_results["Current_Ratio_Prediction"] = (
    validation_data["Current_Log_Ratio"].values
)

commodity_results = (
    validation_results
    .groupby("Commodity")
    .apply(
        lambda group: pd.Series({
            "Observations": len(group),

            "RF_MAE": mean_absolute_error(
                group[TARGET_COLUMN],
                group["RF_Prediction"]
            ),

            "Baseline_MAE": mean_absolute_error(
                group[TARGET_COLUMN],
                group["Current_Ratio_Prediction"]
            ),

            "RF_RMSE": np.sqrt(mean_squared_error(
                group[TARGET_COLUMN],
                group["RF_Prediction"]
            )),

            "Baseline_RMSE": np.sqrt(mean_squared_error(
                group[TARGET_COLUMN],
                group["Current_Ratio_Prediction"]
            ))
        })
    )
    .reset_index()
)

commodity_results["MAE_Improvement_Pct"] = (
    100 * (
        commodity_results["Baseline_MAE"]
        - commodity_results["RF_MAE"]
    ) / commodity_results["Baseline_MAE"]
)

commodity_results["RMSE_Improvement_Pct"] = (
    100 * (
        commodity_results["Baseline_RMSE"]
        - commodity_results["RF_RMSE"]
    ) / commodity_results["Baseline_RMSE"]
)

commodity_results = commodity_results.sort_values(
    "MAE_Improvement_Pct",
    ascending=False
)

display(commodity_results.round(4))

,Commodity,Observations,RF_MAE,Baseline_MAE,RF_RMSE,Baseline_RMSE,MAE_Improvement_Pct,RMSE_Improvement_Pct
25,Onion Green,3911.0,0.2702,0.4579,0.4760,0.9215,40.9953,48.3518
23,Mint (Pudina),1254.0,0.6817,1.1548,1.1902,1.9801,40.9687,39.8922
12,Cowpea (Lobia/Karamani),608.0,0.0700,0.0966,0.1058,0.1545,27.6113,31.4994
39,Yam (Ratalu),1449.0,0.0795,0.1062,0.1054,0.1554,25.1439,32.1455
38,Turnip,1912.0,0.1346,0.1788,0.1840,0.2430,24.7217,24.2747
22,Methi (Leaves),6888.0,0.1984,0.2613,0.3027,0.4847,24.0785,37.5591
0,Amaranthus,13116.0,0.0679,0.0894,0.0983,0.1367,24.0401,28.1259
33,Sweet Potato,4645.0,0.1088,0.1425,0.1488,0.2005,23.6243,25.7558
35,Tapioca,10305.0,0.0457,0.0592,0.0697,0.0949,22.7822,26.5831
32,Surat Beans (Papadi),163.0,0.1910,0.2460,0.2471,0.3490,22.3339,29.2110


In [29]:
from scipy.stats import spearmanr

ranking_data = validation_data[
    SERIES_KEYS + ["Arrival_Date", TARGET_COLUMN]
].copy()

ranking_data["RF_Prediction"] = rf_val_predictions

ranking_results = []

for (commodity, date), group in ranking_data.groupby(
    ["Commodity", "Arrival_Date"]
):
    # Require at least 5 comparable market series
    if len(group) < 5:
        continue

    correlation, _ = spearmanr(
        group["RF_Prediction"],
        group[TARGET_COLUMN]
    )

    if np.isfinite(correlation):
        ranking_results.append({
            "Commodity": commodity,
            "Arrival_Date": date,
            "Market_Count": len(group),
            "Spearman_Correlation": correlation
        })

ranking_results = pd.DataFrame(ranking_results)

commodity_ranking_quality = (
    ranking_results
    .groupby("Commodity")
    .agg(
        Evaluation_Dates=("Spearman_Correlation", "count"),
        Median_Spearman=("Spearman_Correlation", "median"),
        Mean_Spearman=("Spearman_Correlation", "mean"),
        Positive_Ranking_Pct=(
            "Spearman_Correlation",
            lambda x: 100 * (x > 0).mean()
        )
    )
    .sort_values("Median_Spearman", ascending=False)
    .reset_index()
)

display(commodity_ranking_quality.round(4))

,Commodity,Evaluation_Dates,Median_Spearman,Mean_Spearman,Positive_Ranking_Pct
0,Potato,366,0.9667,0.9632,100.0000
1,Carrot,366,0.9649,0.9626,100.0000
2,Beetroot,366,0.9634,0.9561,100.0000
3,Ginger (Green),366,0.9626,0.9595,100.0000
4,Amaranthus,366,0.9551,0.9485,100.0000
5,Cabbage,366,0.9509,0.9480,100.0000
6,Bitter gourd,366,0.9507,0.9472,100.0000
7,Green Chilli,366,0.9502,0.9456,100.0000
8,Tapioca,366,0.9460,0.9366,100.0000
9,Brinjal,366,0.9420,0.9391,100.0000


In [30]:
from scipy.stats import spearmanr

ranking_check = validation_data[
    ["Commodity", "Variety", "Grade", "Arrival_Date",
     "Current_Log_Ratio", TARGET_COLUMN]
].copy()

ranking_check["RF_Prediction"] = rf_val_predictions

ranking_groups = []

GROUP_COLUMNS = [
    "Commodity",
    "Variety",
    "Grade",
    "Arrival_Date"
]

for group_key, group in ranking_check.groupby(GROUP_COLUMNS):

    if len(group) < 5:
        continue

    rf_corr, _ = spearmanr(
        group["RF_Prediction"],
        group[TARGET_COLUMN]
    )

    current_corr, _ = spearmanr(
        group["Current_Log_Ratio"],
        group[TARGET_COLUMN]
    )

    if np.isfinite(rf_corr) and np.isfinite(current_corr):
        ranking_groups.append({
            "Commodity": group_key[0],
            "Variety": group_key[1],
            "Grade": group_key[2],
            "Arrival_Date": group_key[3],
            "Market_Count": len(group),
            "RF_Spearman": rf_corr,
            "Current_Ratio_Spearman": current_corr
        })

ranking_check_results = pd.DataFrame(ranking_groups)

ranking_summary = (
    ranking_check_results
    .groupby("Commodity")
    .agg(
        Evaluation_Groups=("RF_Spearman", "count"),
        RF_Median_Spearman=("RF_Spearman", "median"),
        Current_Median_Spearman=(
            "Current_Ratio_Spearman", "median"
        ),
        RF_Positive_Pct=(
            "RF_Spearman",
            lambda x: 100 * (x > 0).mean()
        ),
        Current_Positive_Pct=(
            "Current_Ratio_Spearman",
            lambda x: 100 * (x > 0).mean()
        )
    )
    .reset_index()
)

ranking_summary["RF_Ranking_Advantage"] = (
    ranking_summary["RF_Median_Spearman"]
    - ranking_summary["Current_Median_Spearman"]
)

display(
    ranking_summary.sort_values(
        "RF_Ranking_Advantage",
        ascending=False
    ).round(4)
)

,Commodity,Evaluation_Groups,RF_Median_Spearman,Current_Median_Spearman,RF_Positive_Pct,Current_Positive_Pct,RF_Ranking_Advantage
1,Beans,679,0.8543,0.8290,99.8527,99.7054,0.0253
33,Sweet Potato,472,0.9226,0.9000,100.0000,99.3644,0.0226
13,Cowpea (Veg),722,0.8861,0.8662,100.0000,99.5845,0.0198
14,Drumstick,727,0.8853,0.8666,100.0000,100.0000,0.0186
26,Peas Wet,690,0.8992,0.8807,100.0000,100.0000,0.0186
5,Bottle gourd,732,0.9102,0.8942,100.0000,100.0000,0.0160
8,Capsicum,732,0.9297,0.9148,100.0000,100.0000,0.0150
30,Spinach,721,0.9225,0.9077,100.0000,100.0000,0.0148
11,Cluster beans,662,0.9219,0.9079,100.0000,100.0000,0.0140
3,Bhindi (Ladies Finger),732,0.9173,0.9037,100.0000,100.0000,0.0136


In [31]:
from scipy.stats import rankdata
from sklearn.metrics import ndcg_score

top_k_results = []

for group_key, group in ranking_check.groupby(GROUP_COLUMNS):

    if len(group) < 5:
        continue

    # Actual future ranking, converted to positive relevance scores
    actual_relevance = rankdata(
        group[TARGET_COLUMN].values,
        method="average"
    )

    rf_scores = group["RF_Prediction"].values
    current_scores = group["Current_Log_Ratio"].values

    rf_ndcg = ndcg_score(
        [actual_relevance],
        [rf_scores],
        k=3
    )

    current_ndcg = ndcg_score(
        [actual_relevance],
        [current_scores],
        k=3
    )

    top_k_results.append({
        "Commodity": group_key[0],
        "Variety": group_key[1],
        "Grade": group_key[2],
        "Arrival_Date": group_key[3],
        "Market_Count": len(group),
        "RF_NDCG_3": rf_ndcg,
        "Current_NDCG_3": current_ndcg
    })

top_k_results = pd.DataFrame(top_k_results)

top_k_summary = (
    top_k_results
    .groupby("Commodity")
    .agg(
        Evaluation_Groups=("RF_NDCG_3", "count"),
        RF_Median_NDCG_3=("RF_NDCG_3", "median"),
        Current_Median_NDCG_3=("Current_NDCG_3", "median"),
        RF_Mean_NDCG_3=("RF_NDCG_3", "mean"),
        Current_Mean_NDCG_3=("Current_NDCG_3", "mean")
    )
    .reset_index()
)

top_k_summary["NDCG_3_Improvement"] = (
    top_k_summary["RF_Median_NDCG_3"]
    - top_k_summary["Current_Median_NDCG_3"]
)

display(
    top_k_summary.sort_values(
        "NDCG_3_Improvement",
        ascending=False
    ).round(4)
)

,Commodity,Evaluation_Groups,RF_Median_NDCG_3,Current_Median_NDCG_3,RF_Mean_NDCG_3,Current_Mean_NDCG_3,NDCG_3_Improvement
38,Turnip,205,0.9901,0.9776,0.9663,0.9566,0.0126
34,Sweet Pumpkin,726,0.9940,0.9835,0.9743,0.9640,0.0105
15,Elephant Yam (Suran),541,0.9761,0.9656,0.9533,0.9409,0.0105
1,Beans,679,0.9584,0.9481,0.9385,0.9229,0.0103
26,Peas Wet,690,0.9669,0.9577,0.9454,0.9353,0.0092
35,Tapioca,674,0.9962,0.9877,0.9831,0.9710,0.0085
0,Amaranthus,711,0.9899,0.9818,0.9787,0.9678,0.0081
39,Yam (Ratalu),151,0.9855,0.9776,0.9652,0.9620,0.0079
22,Methi (Leaves),493,0.9817,0.9742,0.9624,0.9570,0.0075
30,Spinach,721,0.9797,0.9725,0.9620,0.9513,0.0072


In [25]:
top3_comparison = []

for group_key, group in ranking_check.groupby(GROUP_COLUMNS):

    if len(group) < 5:
        continue

    # Top 3 markets selected by each method
    rf_top3 = group.nlargest(3, "RF_Prediction")
    current_top3 = group.nlargest(3, "Current_Log_Ratio")

    # Actual future performance of selected markets
    rf_actual = rf_top3[TARGET_COLUMN].mean()
    current_actual = current_top3[TARGET_COLUMN].mean()

    # Best possible top 3 using actual future outcomes
    actual_top3 = group.nlargest(3, TARGET_COLUMN)
    ideal_actual = actual_top3[TARGET_COLUMN].mean()

    top3_comparison.append({
        "Commodity": group_key[0],
        "Variety": group_key[1],
        "Grade": group_key[2],
        "Arrival_Date": group_key[3],
        "Market_Count": len(group),
        "RF_Top3_Actual_Premium": rf_actual,
        "Current_Top3_Actual_Premium": current_actual,
        "Ideal_Top3_Actual_Premium": ideal_actual
    })

top3_comparison = pd.DataFrame(top3_comparison)

top3_summary = (
    top3_comparison
    .groupby("Commodity")
    .agg(
        Evaluation_Groups=("RF_Top3_Actual_Premium", "count"),
        RF_Top3_Mean_Premium=("RF_Top3_Actual_Premium", "mean"),
        Current_Top3_Mean_Premium=(
            "Current_Top3_Actual_Premium", "mean"
        ),
        Ideal_Top3_Mean_Premium=(
            "Ideal_Top3_Actual_Premium", "mean"
        )
    )
    .reset_index()
)

top3_summary["RF_Improvement"] = (
    top3_summary["RF_Top3_Mean_Premium"]
    - top3_summary["Current_Top3_Mean_Premium"]
)

display(
    top3_summary.sort_values(
        "RF_Improvement", ascending=False
    ).round(4)
)

print(
    "Percentage of evaluation groups where RF selects "
    "a better-performing top three:",
    round(
        100 * (
            top3_comparison["RF_Top3_Actual_Premium"]
            > top3_comparison["Current_Top3_Actual_Premium"]
        ).mean(),
        2
    ),
    "%"
)

,Commodity,Evaluation_Groups,RF_Top3_Mean_Premium,Current_Top3_Mean_Premium,Ideal_Top3_Mean_Premium,RF_Improvement
5,Bottle gourd,732,0.9600,0.9342,1.0016,0.0258
7,Cabbage,732,1.1015,1.0800,1.1376,0.0215
29,Pumpkin,732,0.8551,0.8369,0.8951,0.0183
30,Spinach,721,0.7347,0.7175,0.7866,0.0172
3,Bhindi (Ladies Finger),732,0.8552,0.8395,0.9061,0.0157
37,Tomato,1830,0.6336,0.6198,0.6872,0.0138
1,Beans,679,0.4083,0.3946,0.4558,0.0137
10,Cauliflower,1442,0.7567,0.7454,0.7907,0.0113
23,Mint (Pudina),27,0.8998,0.8888,0.9356,0.0111
0,Amaranthus,711,0.4764,0.4655,0.4934,0.0109


Percentage of evaluation groups where RF selects a better-performing top three: 24.25 %


In [26]:
top3_comparison["RF_Improvement"] = (
    top3_comparison["RF_Top3_Actual_Premium"]
    - top3_comparison["Current_Top3_Actual_Premium"]
)

print("Overall group-level comparison")
print(
    top3_comparison["RF_Improvement"]
    .describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90])
    .round(4)
)

print("\nGroup-level outcomes")
print(
    pd.Series({
        "RF wins (%)": 100 * (
            top3_comparison["RF_Improvement"] > 0
        ).mean(),
        "Ties (%)": 100 * (
            top3_comparison["RF_Improvement"] == 0
        ).mean(),
        "RF losses (%)": 100 * (
            top3_comparison["RF_Improvement"] < 0
        ).mean(),
        "Mean improvement": top3_comparison["RF_Improvement"].mean(),
        "Median improvement": top3_comparison["RF_Improvement"].median()
    }).round(4)
)

print("\nCommodity-level group win rates")
commodity_win_rates = (
    top3_comparison
    .groupby("Commodity")
    .agg(
        Groups=("RF_Improvement", "count"),
        Win_Rate_Pct=(
            "RF_Improvement",
            lambda x: 100 * (x > 0).mean()
        ),
        Mean_Improvement=("RF_Improvement", "mean"),
        Median_Improvement=("RF_Improvement", "median")
    )
    .reset_index()
    .sort_values("Win_Rate_Pct")
)

display(commodity_win_rates.round(4))

Overall group-level comparison
count    31455.0000
mean         0.0090
std          0.0533
min         -0.4517
10%         -0.0182
25%          0.0000
50%          0.0000
75%          0.0000
90%          0.0561
max          0.7932
Name: RF_Improvement, dtype: float64

Group-level outcomes
RF wins (%)           24.2505
Ties (%)              58.8364
RF losses (%)         16.9131
Mean improvement       0.0090
Median improvement     0.0000
dtype: float64

Commodity-level group win rates


,Commodity,Groups,Win_Rate_Pct,Mean_Improvement,Median_Improvement
12,Cowpea (Lobia/Karamani),13,0.0000,-0.0031,0.0
20,Green Peas,289,11.0727,0.0035,0.0
39,Yam (Ratalu),151,11.2583,-0.0007,0.0
21,Indian Beans (Seam),291,13.4021,0.0025,0.0
32,Surat Beans (Papadi),7,14.2857,0.0063,0.0
25,Onion Green,427,14.5199,0.0028,0.0
34,Sweet Pumpkin,726,16.6667,0.0064,0.0
33,Sweet Potato,472,17.3729,0.0044,0.0
28,Potato,3119,17.8903,0.0053,0.0
11,Cluster beans,662,18.1269,0.0046,0.0


In [27]:
improvement = top3_comparison["RF_Improvement"]

print("Average improvement when RF wins:")
print(round(improvement[improvement > 0].mean(), 4))

print("\nAverage loss when RF loses:")
print(round(improvement[improvement < 0].mean(), 4))

print("\nMedian improvement among wins:")
print(round(improvement[improvement > 0].median(), 4))

print("\nMedian loss among losses:")
print(round(improvement[improvement < 0].median(), 4))

print("\nShare of groups with a meaningful difference:")
for threshold in [0.01, 0.025, 0.05, 0.10]:
    print(
        f"|Improvement| > {threshold}: "
        f"{100 * (improvement.abs() > threshold).mean():.2f}%"
    )

print("\nRanking-check columns:")
print(ranking_check.columns.tolist())

Average improvement when RF wins:
0.0647

Average loss when RF loses:
-0.0397

Median improvement among wins:
0.0428

Median loss among losses:
-0.0252

Share of groups with a meaningful difference:
|Improvement| > 0.01: 32.11%
|Improvement| > 0.025: 24.52%
|Improvement| > 0.05: 15.54%
|Improvement| > 0.1: 6.52%

Ranking-check columns:
['Commodity', 'Variety', 'Grade', 'Arrival_Date', 'Current_Log_Ratio', 'Target_7D_Median_Log_Ratio', 'RF_Prediction']


In [28]:
ranking_check = validation_data[
    [
        "Commodity",
        "Variety",
        "Grade",
        "Arrival_Date",
        "State",
        "District",
        "Market",
        "Current_Log_Ratio",
        TARGET_COLUMN
    ]
].copy()

ranking_check["RF_Prediction"] = rf_val_predictions

print("Shape:", ranking_check.shape)
print("Missing market identifiers:")
print(
    ranking_check[
        ["State", "District", "Market"]
    ].isna().sum()
)

display(ranking_check.head())

Shape: (1613032, 10)
Missing market identifiers:
State       0
District    0
Market      0
dtype: int64


,Commodity,Variety,Grade,Arrival_Date,State,District,Market,Current_Log_Ratio,Target_7D_Median_Log_Ratio,RF_Prediction
5165396,Cabbage,Cabbage,FAQ,2024-01-01,Haryana,Kurukshetra,Ladwa,-0.024693,-0.072223,-0.028156
5165397,Beetroot,Beetroot,FAQ,2024-01-01,Karnataka,Kolar,Chintamani,-0.113329,-0.336472,-0.388633
5165398,Pumpkin,Other,FAQ,2024-01-01,Uttrakhand,Haridwar,Roorkee,-1.144223,-1.098612,-1.029310
5165399,Bottle gourd,Bottle Gourd,FAQ,2024-01-01,Kerala,Palakad,Palakkad,0.678441,0.601210,0.661815
5165400,Potato,Other,FAQ,2024-01-01,Himachal Pradesh,Kangra,Kangra,0.262364,0.318288,0.252274


In [29]:
selection_overlap = []

for group_key, group in ranking_check.groupby(GROUP_COLUMNS):

    if len(group) < 5:
        continue

    rf_top3 = group.nlargest(3, "RF_Prediction")
    current_top3 = group.nlargest(3, "Current_Log_Ratio")

    rf_markets = set(
        zip(rf_top3["State"], rf_top3["District"], rf_top3["Market"])
    )

    current_markets = set(
        zip(
            current_top3["State"],
            current_top3["District"],
            current_top3["Market"]
        )
    )

    overlap = len(rf_markets.intersection(current_markets))

    selection_overlap.append({
        "Commodity": group_key[0],
        "Variety": group_key[1],
        "Grade": group_key[2],
        "Arrival_Date": group_key[3],
        "Market_Count": len(group),
        "Shared_Markets": overlap,
        "Different_Markets": 3 - overlap
    })

selection_overlap = pd.DataFrame(selection_overlap)

print("Top-three selection overlap")
display(
    selection_overlap["Shared_Markets"]
    .value_counts()
    .sort_index()
    .rename_axis("Shared_Markets")
    .to_frame("Evaluation_Groups")
)

print("\nPercentage distribution")
display(
    (
        100 * selection_overlap["Shared_Markets"]
        .value_counts(normalize=True)
        .sort_index()
    ).round(2).rename("Percentage").to_frame()
)

Top-three selection overlap


,Evaluation_Groups
Shared_Markets,
0,25
1,883
2,11400
3,19147



Percentage distribution


,Percentage
Shared_Markets,
0,0.08
1,2.81
2,36.24
3,60.87


In [32]:
# Recreate the untouched 2025 test dataset

test_data = model_data[
    model_data["Split"] == "Test"
].copy()

print("Test shape:", test_data.shape)
print("Date range:",
      test_data["Arrival_Date"].min(),
      "to",
      test_data["Arrival_Date"].max())

print("Unique commodities:", test_data["Commodity"].nunique())
print("Missing target values:",
      test_data[TARGET_COLUMN].isna().sum())

Test shape: (1116517, 25)
Date range: 2025-01-01 00:00:00 to 2025-10-24 00:00:00
Unique commodities: 40
Missing target values: 0


In [33]:
print("rf_model exists:", "rf_model" in globals())
print("rf_test_predictions exists:", "rf_test_predictions" in globals())

if "rf_model" in globals():
    print("Model type:", type(rf_model))

if "rf_test_predictions" in globals():
    print("Prediction count:", len(rf_test_predictions))

rf_model exists: True
rf_test_predictions exists: False
Model type: <class 'sklearn.ensemble._forest.RandomForestRegressor'>


In [34]:
print("Feature columns:")
print(FEATURE_COLUMNS)

print("\nRF model expected features:")
print(rf_model.n_features_in_)

print("\nTest feature missing values:")
display(
    test_data[FEATURE_COLUMNS]
    .isna()
    .sum()
    .to_frame("Missing_Count")
    .query("Missing_Count > 0")
)

Feature columns:
['Current_Log_Ratio', 'Lag_1_Log_Ratio', 'Lag_3_Log_Ratio', 'Lag_7_Log_Ratio', 'Rolling_Median_7', 'Rolling_Median_30', 'Rolling_Mean_7', 'Rolling_Std_7', 'Log_Price_Change_1', 'Log_Price_Change_7', 'Days_Since_Previous_Report', 'Peer_Market_Count', 'Source_Records', 'Month', 'Day_Of_Year']

RF model expected features:
15

Test feature missing values:


,Missing_Count


In [35]:
# Generate test predictions using the already-fitted model

X_test = test_data[FEATURE_COLUMNS]

rf_test_predictions = rf_model.predict(X_test)

print("Test predictions generated:", len(rf_test_predictions))
print("Expected observations:", len(test_data))
print("Any non-finite predictions:",
      not np.isfinite(rf_test_predictions).all())

Test predictions generated: 1116517
Expected observations: 1116517
Any non-finite predictions: False


In [38]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

actual = test_data[TARGET_COLUMN]

baseline_predictions = test_data["Current_Log_Ratio"]

test_results = pd.DataFrame([
    {
        "Model": "Current Ratio Baseline",
        "MAE": mean_absolute_error(actual, baseline_predictions),
        "RMSE": np.sqrt(
            mean_squared_error(actual, baseline_predictions)
        )
    },
    {
        "Model": "Random Forest",
        "MAE": mean_absolute_error(actual, rf_test_predictions),
        "RMSE": np.sqrt(
            mean_squared_error(actual, rf_test_predictions)
        )
    }
])

display(test_results.round(4))

baseline_mae = test_results.loc[
    test_results["Model"] == "Current Ratio Baseline", "MAE"
].iloc[0]

rf_mae = test_results.loc[
    test_results["Model"] == "Random Forest", "MAE"
].iloc[0]

baseline_rmse = test_results.loc[
    test_results["Model"] == "Current Ratio Baseline", "RMSE"
].iloc[0]

rf_rmse = test_results.loc[
    test_results["Model"] == "Random Forest", "RMSE"
].iloc[0]

print("MAE improvement (%):",
      round((baseline_mae - rf_mae) / baseline_mae * 100, 2))

print("RMSE improvement (%):",
      round((baseline_rmse - rf_rmse) / baseline_rmse * 100, 2))

,Model,MAE,RMSE
0,Current Ratio Baseline,0.1037,0.1668
1,Random Forest,0.0913,0.1398


MAE improvement (%): 12.0
RMSE improvement (%): 16.21


In [39]:
test_ranking_check = test_data[
    [
        "Commodity",
        "Variety",
        "Grade",
        "Arrival_Date",
        "State",
        "District",
        "Market",
        "Current_Log_Ratio",
        TARGET_COLUMN
    ]
].copy()

test_ranking_check["RF_Prediction"] = rf_test_predictions

print("Test ranking dataset:", test_ranking_check.shape)
print("Unique commodities:",
      test_ranking_check["Commodity"].nunique())

print("Missing values:")
display(
    test_ranking_check[
        ["Current_Log_Ratio", TARGET_COLUMN, "RF_Prediction"]
    ].isna().sum()
)

Test ranking dataset: (1116517, 10)
Unique commodities: 40
Missing values:


Current_Log_Ratio             0
Target_7D_Median_Log_Ratio    0
RF_Prediction                 0
dtype: int64

In [40]:
GROUP_COLUMNS = [
    "Commodity",
    "Variety",
    "Grade",
    "Arrival_Date"
]

test_top3_comparison = []

for group_key, group in test_ranking_check.groupby(GROUP_COLUMNS):

    if len(group) < 5:
        continue

    rf_top3 = group.nlargest(3, "RF_Prediction")
    baseline_top3 = group.nlargest(3, "Current_Log_Ratio")
    ideal_top3 = group.nlargest(3, TARGET_COLUMN)

    test_top3_comparison.append({
        "Commodity": group_key[0],
        "Variety": group_key[1],
        "Grade": group_key[2],
        "Arrival_Date": group_key[3],
        "Market_Count": len(group),
        "RF_Top3_Actual": rf_top3[TARGET_COLUMN].mean(),
        "Baseline_Top3_Actual": baseline_top3[TARGET_COLUMN].mean(),
        "Ideal_Top3_Actual": ideal_top3[TARGET_COLUMN].mean()
    })

test_top3_comparison = pd.DataFrame(test_top3_comparison)

test_top3_comparison["RF_Improvement"] = (
    test_top3_comparison["RF_Top3_Actual"]
    - test_top3_comparison["Baseline_Top3_Actual"]
)

print("Evaluation groups:", len(test_top3_comparison))

print("\nOverall test results:")
display(
    test_top3_comparison[
        [
            "RF_Top3_Actual",
            "Baseline_Top3_Actual",
            "Ideal_Top3_Actual",
            "RF_Improvement"
        ]
    ].mean().to_frame("Mean").round(4)
)

print("\nRF wins (%):",
      round(100 * (
          test_top3_comparison["RF_Improvement"] > 0
      ).mean(), 2))

print("Ties (%):",
      round(100 * (
          test_top3_comparison["RF_Improvement"] == 0
      ).mean(), 2))

print("RF losses (%):",
      round(100 * (
          test_top3_comparison["RF_Improvement"] < 0
      ).mean(), 2))

Evaluation groups: 23626

Overall test results:


,Mean
RF_Top3_Actual,0.6272
Baseline_Top3_Actual,0.6181
Ideal_Top3_Actual,0.6581
RF_Improvement,0.0091



RF wins (%): 23.12
Ties (%): 61.24
RF losses (%): 15.64


In [41]:
test_top3_summary = (
    test_top3_comparison
    .groupby("Commodity")
    .agg(
        Evaluation_Groups=("RF_Improvement", "size"),
        RF_Mean=("RF_Top3_Actual", "mean"),
        Baseline_Mean=("Baseline_Top3_Actual", "mean"),
        Mean_Improvement=("RF_Improvement", "mean"),
        Median_Improvement=("RF_Improvement", "median"),
        Win_Rate=("RF_Improvement",
                  lambda x: 100 * (x > 0).mean()),
        Tie_Rate=("RF_Improvement",
                  lambda x: 100 * (x == 0).mean()),
        Loss_Rate=("RF_Improvement",
                   lambda x: 100 * (x < 0).mean())
    )
    .reset_index()
    .sort_values("Mean_Improvement", ascending=False)
)

display(test_top3_summary.round(4))

print("\nOverall improvement distribution:")
display(
    test_top3_comparison["RF_Improvement"]
    .describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    )
    .to_frame("RF_Improvement")
    .round(4)
)

,Commodity,Evaluation_Groups,RF_Mean,Baseline_Mean,Mean_Improvement,Median_Improvement,Win_Rate,Tie_Rate,Loss_Rate
5,Bottle gourd,592,1.0074,0.9824,0.0250,0.0000,31.7568,56.4189,11.8243
28,Pumpkin,594,0.8734,0.8519,0.0216,0.0000,29.4613,50.8418,19.6970
7,Cabbage,592,1.2672,1.2477,0.0195,0.0000,30.7432,52.3649,16.8919
12,Cowpea (Lobia/Karamani),2,0.1777,0.1613,0.0163,0.0163,50.0000,50.0000,0.0000
36,Tomato,1473,0.6700,0.6546,0.0154,0.0000,28.7169,53.6320,17.6511
2,Beetroot,583,0.5988,0.5847,0.0141,0.0000,31.7324,51.9726,16.2950
19,Green Chilli,592,1.1541,1.1406,0.0136,0.0000,25.3378,58.9527,15.7095
10,Cauliflower,1112,0.9008,0.8875,0.0133,0.0000,21.6727,67.3561,10.9712
24,Onion Green,139,0.3588,0.3457,0.0131,0.0000,23.0216,71.9424,5.0360
18,Ginger (Green),586,0.7444,0.7317,0.0127,0.0000,27.6451,52.5597,19.7952



Overall improvement distribution:


,RF_Improvement
count,23626.0000
mean,0.0091
std,0.0504
min,-0.4946
10%,-0.0140
25%,0.0000
50%,0.0000
75%,0.0000
90%,0.0528
max,0.6920


In [42]:
improvements = test_top3_comparison["RF_Improvement"]

wins = improvements[improvements > 0]
losses = improvements[improvements < 0]
ties = improvements[improvements == 0]

print("WIN / TIE / LOSS ANALYSIS")
print("-" * 40)

print("Winning groups:", len(wins))
print("Average win:", round(wins.mean(), 4))
print("Median win:", round(wins.median(), 4))

print("\nLosing groups:", len(losses))
print("Average loss:", round(losses.mean(), 4))
print("Median loss:", round(losses.median(), 4))

print("\nTied groups:", len(ties))

print("\nContribution to total improvement:")
print(
    "Total positive improvement:",
    round(wins.sum(), 4)
)
print(
    "Total negative improvement:",
    round(losses.sum(), 4)
)
print(
    "Net improvement:",
    round(improvements.sum(), 4)
)

WIN / TIE / LOSS ANALYSIS
----------------------------------------
Winning groups: 5463
Average win: 0.0645
Median win: 0.044

Losing groups: 3695
Average loss: -0.0371
Median loss: -0.0232

Tied groups: 14468

Contribution to total improvement:
Total positive improvement: 352.2089
Total negative improvement: -137.1449
Net improvement: 215.064


In [43]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

extra_trees_model = ExtraTreesRegressor(
    n_estimators=200,
    max_features=0.8,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42
)

X_train = train_data[FEATURE_COLUMNS]
y_train = train_data[TARGET_COLUMN]

X_val = validation_data[FEATURE_COLUMNS]
y_val = validation_data[TARGET_COLUMN]

extra_trees_model.fit(X_train, y_train)

et_val_predictions = extra_trees_model.predict(X_val)

et_mae = mean_absolute_error(y_val, et_val_predictions)
et_rmse = np.sqrt(mean_squared_error(y_val, et_val_predictions))

print("Extra Trees Validation Results")
print("MAE:", round(et_mae, 4))
print("RMSE:", round(et_rmse, 4))

MemoryError: Unable to allocate 19.7 MiB for an array with shape (5165396,) and data type float32